# SAP Security Logs — Exploratory Data Analysis

This notebook provides a quick overview of the security log data:
- Load sample/mock logs
- Basic statistics and distributions
- Feature engineering preview
- Anomaly score visualization

**Usage:** Run `make mock` first to generate sample data, then run this notebook.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Load mock or real logs
DATA_DIR = Path("../data/samples")
csv_path = DATA_DIR / "sample_logs.csv"
if not csv_path.exists():
    csv_path = DATA_DIR / "mock_logs.csv"

df = pd.read_csv(csv_path, parse_dates=["datetime"])
print(f"Loaded {len(df):,} rows from {csv_path.name}")
df.head()

## Basic Statistics

In [ ]:
print(f"Time range: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"Unique IPs: {df['source_ip'].nunique()}")
print(f"Event types: {df['event_type'].nunique() if 'event_type' in df.columns else 'N/A'}")
print()
df.describe()

## Request Volume Over Time

In [ ]:
timeline = df.set_index("datetime").resample("1min").size().reset_index(name="requests")
fig = px.area(timeline, x="datetime", y="requests", title="Requests per Minute")
fig.show()

## Top Source IPs

In [ ]:
top_ips = df["source_ip"].value_counts().head(20).reset_index()
top_ips.columns = ["source_ip", "count"]
fig = px.bar(top_ips, x="count", y="source_ip", orientation="h",
             title="Top 20 Source IPs by Request Count")
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

## Feature Engineering Preview

In [ ]:
from src.model.features import extract_features

features_df = extract_features(df)
print(f"Feature matrix: {features_df.shape}")
features_df.describe()

## Feature Distributions

In [ ]:
numeric_cols = features_df.select_dtypes(include=[np.number]).columns.tolist()
# Remove source_ip if present
plot_cols = [c for c in numeric_cols if c != "source_ip"][:6]

for col in plot_cols:
    fig = px.histogram(features_df, x=col, title=f"Distribution: {col}", nbins=50)
    fig.show()

## Anomaly Scoring (if model exists)

In [ ]:
from src.common.time_utils import utcnow

try:
    from src.model.predict import predict
    scored = predict(features_df, ingested_at=utcnow(), batch_min_log_time=None)
    print(f"Scored {len(scored)} IPs")
    print(f"Anomalies: {scored['is_anomaly'].sum()}")
    print()
    print(scored[["source_ip", "anomaly_score", "threat_level", "is_anomaly"]].sort_values("anomaly_score").head(20))
except Exception as e:
    print(f"Model not available: {e}")
    print("Run 'make train' first to train a model.")
    scored = None

In [ ]:
if scored is not None and not scored.empty:
    fig = px.histogram(scored, x="anomaly_score", color="threat_level",
                       title="Anomaly Score Distribution by Threat Level",
                       color_discrete_map={"high": "red", "medium": "orange", "low": "green"},
                       nbins=50)
    fig.show()

    fig2 = px.scatter(scored, x="total_requests", y="anomaly_score",
                      color="threat_level", hover_data=["source_ip"],
                      title="Anomaly Score vs Total Requests",
                      color_discrete_map={"high": "red", "medium": "orange", "low": "green"})
    fig2.show()